## Decision Tree Classifier

Classifies by recursively partitioning data using feature-based conditions, producing a tree with decision nodes and leaves representing class labels.

<br>

<p align="center">
<img src="visualizations/decision_tree.png" width="600">
</p>

**Training Steps:**

1. Begin with all data at the root node.
2. At each node, select the split (feature and threshold/category) that **maximizes impurity reduction**.
3. Partition the data according to the chosen split and recurse on each subset.
4. Stop when nodes are pure, a maximum depth is reached, or no split improves impurity.

---

**Impurity Measures:**

* **Gini impurity:**

$$
Gini = 1 - \sum_{k=1}^{K} p_k^2
$$

* **Entropy:**

$$
Entropy = -\sum_{k=1}^{K} p_k \log_2(p_k)
$$

* **Information Gain:**

$$
IG = I(parent) - \sum_{j} \frac{n_j}{n} I(child_j)
$$

---

**Handling Features:**

* Continuous: consider splits of the form $x \leq t$.
* Categorical: evaluate partitions of categories.

---

**Prediction:**
Classify a sample by traversing from the root, following splits until reaching a leaf, then output the leaf’s class label.


---

**Pros:**
Interpretable; handles mixed data types; no feature scaling required.

**Cons:**
Prone to overfitting; greedy splits may not be globally optimal; high variance without ensembles.


In [50]:
import numpy as np
from tqdm import tqdm
from cifar10.unpickle import get_all_data, get_test_data

In [51]:
# 1) Load
x_train, y_train = get_all_data()
x_test, y_test = get_test_data()

y_train = np.array(y_train)
y_test = np.array(y_test)

n_samples, h, w, c = x_train.shape  # h=32, w=32, c=3
x_train = x_train.reshape(n_samples, h * w * c)  # flatten to (N, 3072)
x_test = x_test.reshape(x_test.shape[0], h * w * c)  # flatten to (N, 3072)

In [52]:
class Node:
    def __init__(self, feature_index=None, threshold=None, left=None, right=None, *, value=None):
        self.feature_index = feature_index
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

    def is_leaf(self):
        return self.value is not None



class DecisionTree:
    def __init__(self, max_depth=10, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    def fit(self, X, y):
        self.root = self._build_tree(X, y)

    def _entropy(self, y):
        classes, counts = np.unique(y, return_counts=True)
        probs = counts / counts.sum()
        return -np.sum(probs * np.log2(probs + 1e-9))

    def _best_split(self, X, y):
        best_gain = -1
        split_idx, split_thresh = None, None
        current_entropy = self._entropy(y)

        n_samples, n_features = X.shape
        feature_indices = np.random.choice(n_features, 50, replace=False)

        for feature in feature_indices:
            thresholds = np.unique(X[:, feature])
            for t in thresholds:
                left_idx = X[:, feature] < t
                right_idx = X[:, feature] >= t
                if len(y[left_idx]) == 0 or len(y[right_idx]) == 0:
                    continue

                left_entropy = self._entropy(y[left_idx])
                right_entropy = self._entropy(y[right_idx])
                p = len(y[left_idx]) / len(y)
                gain = current_entropy - (p * left_entropy + (1 - p) * right_entropy)

                if gain > best_gain:
                    best_gain = gain
                    split_idx = feature
                    split_thresh = t

        return split_idx, split_thresh

    def _build_tree(self, X, y, depth=0):
        n_samples, n_features = X.shape
        n_labels = len(np.unique(y))

        if (depth >= self.max_depth or n_labels == 1 or n_samples < self.min_samples_split):
            leaf_value = self._most_common_label(y)
            return Node(value=leaf_value)

        feat_idx, threshold = self._best_split(X, y)
        if feat_idx is None:
            return Node(value=self._most_common_label(y))

        left_idxs = X[:, feat_idx] < threshold
        right_idxs = X[:, feat_idx] >= threshold
        left = self._build_tree(X[left_idxs], y[left_idxs], depth + 1)
        right = self._build_tree(X[right_idxs], y[right_idxs], depth + 1)
        return Node(feature_index=feat_idx, threshold=threshold, left=left, right=right)

    def _most_common_label(self, y):
        labels, counts = np.unique(y, return_counts=True)
        return labels[np.argmax(counts)]

    def _traverse(self, x, node):
        if node.is_leaf():
            return node.value
        if x[node.feature_index] < node.threshold:
            return self._traverse(x, node.left)
        return self._traverse(x, node.right)

    def predict(self, X):
        return np.array([self._traverse(x, self.root) for x in X])


In [53]:
tree = DecisionTree(max_depth=10)
tree.fit(x_train, y_train)

preds = tree.predict(x_test)
acc = np.mean(preds == y_test)
print(f"Test Accuracy: {acc:.4f}")

Test Accuracy: 0.2897


## Random Forest Classifier

Ensemble of decision trees, combines the predictions of multiple trees to improve generalization and reduce overfitting.


<br>

<p align="center">
<img src="visualizations/random_forest.png" width="1000">
</p>


**Core Idea:**
Train multiple decision trees on different random subsets of the data and features. Combine their predictions (e.g., by majority vote) to obtain the final output.


* **Bootstrap Aggregating (Bagging):**
  Each tree is trained on a random sample (with replacement) of the training data.

* **Feature Randomness:**
  At each split, a random subset of features is considered—introducing more diversity among trees.

---

**Prediction:**
Each tree outputs a class label; the forest returns the **majority vote** across all trees.


In [54]:
from collections import Counter

class RandomForest:
    def __init__(self, n_estimators=10, max_depth=10, min_samples_split=2):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.trees = []

    def fit(self, X, y):
        self.trees = []
        n_samples = X.shape[0]

        for _ in tqdm(range(self.n_estimators), desc="Training trees"):
            indices = np.random.choice(n_samples, n_samples, replace=True)
            X_sample = X[indices]
            y_sample = y[indices]

            tree = DecisionTree(max_depth=self.max_depth, min_samples_split=self.min_samples_split)
            tree.fit(X_sample, y_sample)
            self.trees.append(tree)

    def predict(self, X):
        tree_preds = np.array([tree.predict(X) for tree in self.trees])
        tree_preds = np.swapaxes(tree_preds, 0, 1)

        y_pred = np.array([Counter(row).most_common(1)[0][0] for row in tree_preds])
        return y_pred


In [ ]:
forest = RandomForest(n_estimators=50, max_depth=10)
forest.fit(x_train, y_train)

preds = forest.predict(x_test)
acc = np.mean(preds == y_test)
print(f"Test Accuracy: {acc:.4f}")

Training trees: 100%|██████████| 50/50 [2:29:06<00:00, 178.94s/it]  


Test Accuracy: 0.3995


: 

TODO: Experiment with data preprocessing. Use dimensionality reduction or extract image descriptors for smaller representation space